In [1]:
from manim import *

In [7]:
%%manim -qh -v WARNING Berakningsformeln
from manim import *
import numpy as np

def slim_h_brace(p_left, p_right, y_offset=-0.28, tick_h=0.07, color=WHITE, stroke_width=1.5):
    """Nätt horisontell måttklam under ett intervall [p_left, p_right]."""
    x_l, y_base = p_left[0], p_left[1]
    x_r = p_right[0]
    y_line = y_base + y_offset
    return VGroup(
        Line([x_l, y_line, 0], [x_r, y_line, 0], color=color, stroke_width=stroke_width),
        Line([x_l, y_line, 0], [x_l, y_line + tick_h, 0], color=color, stroke_width=stroke_width),
        Line([x_r, y_line, 0], [x_r, y_line + tick_h, 0], color=color, stroke_width=stroke_width),
    )

def slim_v_brace(p_bot, p_top, x_offset=-0.62, tick_w=0.07, color=WHITE, stroke_width=1.5):
    """Nätt vertikal måttklam till vänster om ett intervall [p_bot, p_top]."""
    x_base = p_bot[0]
    y_b, y_t = p_bot[1], p_top[1]
    x_line = x_base + x_offset
    return VGroup(
        Line([x_line, y_b, 0], [x_line, y_t, 0], color=color, stroke_width=stroke_width),
        Line([x_line, y_b, 0], [x_line + tick_w, y_b, 0], color=color, stroke_width=stroke_width),
        Line([x_line, y_t, 0], [x_line + tick_w, y_t, 0], color=color, stroke_width=stroke_width),
    )

class Berakningsformeln(Scene):
    def construct(self):
        # 0. Rubrik som syns direkt i första bildrutan (innan play trycks)
        title = Text("Integralkalkylens huvudsats", font_size=26, color=WHITE).to_corner(UR, buff=0.35)
        title_line = Line(title.get_left(), title.get_right(), color=GRAY_B, stroke_width=1.5).next_to(
            title, DOWN, buff=0.08
        )
        self.add(title, title_line)

        # 1. Kraftig krökning så sekanterna syns tydligt
        def F(x):
            return 0.25 * np.exp(0.65 * x) + 0.3

        def f(x):
            return 0.25 * 0.65 * np.exp(0.65 * x)

        a, b = 1.0, 4.0
        n_steps = 2
        dx = (b - a) / n_steps

        # 2. Koordinatsystem utsträckta i y-led med slanka pilar
        ax_cfg = {
            "include_tip": True,
            "stroke_width": 2,
            "tip_width": 0.12,
            "tip_height": 0.12,
            "include_ticks": False
        }

        ax_top = Axes(
            x_range=[0, 4.5, 1], y_range=[0, 4.2, 1],
            x_length=5.0, y_length=2.7,
            axis_config=ax_cfg
        ).move_to(LEFT * 2.55 + UP * 1.85)

        ax_bot = Axes(
            x_range=[0, 4.5, 1], y_range=[0, 2.5, 1],
            x_length=5.0, y_length=2.25,
            axis_config=ax_cfg
        ).move_to(LEFT * 2.55 + DOWN * 1.55)

        # Axelrubriker centrerade rakt ovanför respektive y-axelpil + x på x-axlarna
        ax_top_y_lbl = MathTex("y = F(x)", font_size=25, color=BLUE).next_to(
            ax_top.y_axis.get_end(), UP, buff=0.14
        )
        ax_top_x_lbl = MathTex("x", font_size=24).next_to(ax_top.x_axis.get_end(), RIGHT, buff=0.08)

        ax_bot_y_lbl = MathTex("y = f(x) = F'(x)", font_size=25, color=YELLOW).next_to(
            ax_bot.y_axis.get_end(), UP, buff=0.14
        )
        ax_bot_x_lbl = MathTex("x", font_size=24).next_to(ax_bot.x_axis.get_end(), RIGHT, buff=0.08)

        # Kurvor
        graph_F = ax_top.plot(F, x_range=[0.4, 4.15], color=BLUE, stroke_width=3)
        graph_f = ax_bot.plot(f, x_range=[0.4, 4.15], color=YELLOW, stroke_width=3)

        # Markera a, mittpunkten (a + dx) och b på båda x-axlarna
        x_mid = a + dx
        x_marks = VGroup()
        for val, name in [(a, "a"), (b, "b")]:
            for ax, func in [(ax_top, F), (ax_bot, f)]:
                tick = ax.x_axis.get_tick(val, size=0.08)
                lbl = MathTex(name, font_size=23).next_to(ax.c2p(val, 0), DOWN, buff=0.09)
                dash = DashedLine(ax.c2p(val, 0), ax.c2p(val, func(val)), color=GRAY, stroke_width=1.5)
                x_marks.add(tick, lbl, dash)

        for ax, func in [(ax_top, F), (ax_bot, f)]:
            x_marks.add(
                ax.x_axis.get_tick(x_mid, size=0.08),
                DashedLine(ax.c2p(x_mid, 0), ax.c2p(x_mid, func(x_mid)), color=GRAY_D, stroke_width=1.2)
            )

        self.play(
            Create(ax_top), Create(ax_bot),
            Write(ax_top_y_lbl), Write(ax_top_x_lbl),
            Write(ax_bot_y_lbl), Write(ax_bot_x_lbl)
        )
        self.play(Create(graph_F), Create(graph_f), FadeIn(x_marks))

        # 3. F(a) och F(b) direkt på övre y-axeln + nätt måttklam med F(b)-F(a)
        pt_a = Dot(ax_top.c2p(a, F(a)), color=WHITE, radius=0.055)
        pt_b = Dot(ax_top.c2p(b, F(b)), color=WHITE, radius=0.055)

        tick_Fa = ax_top.y_axis.get_tick(F(a), size=0.08)
        tick_Fb = ax_top.y_axis.get_tick(F(b), size=0.08)
        lbl_Fa = MathTex("F(a)", font_size=22).next_to(ax_top.c2p(0, F(a)), LEFT, buff=0.1)
        lbl_Fb = MathTex("F(b)", font_size=22).next_to(ax_top.c2p(0, F(b)), LEFT, buff=0.1)

        dash_Fa = DashedLine(ax_top.c2p(0, F(a)), ax_top.c2p(a, F(a)), color=GRAY_B, stroke_width=1.5)
        dash_Fb = DashedLine(ax_top.c2p(0, F(b)), ax_top.c2p(b, F(b)), color=GRAY_B, stroke_width=1.5)

        brace_diff = slim_v_brace(
            ax_top.c2p(0, F(a)), ax_top.c2p(0, F(b)), x_offset=-0.68, tick_w=0.06
        )
        lbl_diff = MathTex("F(b)-F(a)", font_size=20).rotate(PI / 2).next_to(brace_diff, LEFT, buff=0.06)

        self.play(FadeIn(pt_a, pt_b, tick_Fa, tick_Fb, lbl_Fa, lbl_Fb))
        self.play(Create(dash_Fa), Create(dash_Fb), Create(brace_diff), Write(lbl_diff))
        self.wait(0.8)

        # 4. De två stegen med sekant, x_1/x_2 i mitten av Δx, nätt måttklam för Δx och lutning på nedre y-axeln
        dy_lines = VGroup()

        for i in range(n_steps):
            x0 = a + i * dx
            x1 = x0 + dx
            x_center = x0 + dx / 2
            y0 = F(x0)
            y1 = F(x1)
            slope = (y1 - y0) / dx

            step_h = Line(ax_top.c2p(x0, y0), ax_top.c2p(x1, y0), color=WHITE, stroke_width=2.2)
            step_v = Line(ax_top.c2p(x1, y0), ax_top.c2p(x1, y1), color=GREEN, stroke_width=4)
            secant = Line(ax_top.c2p(x0, y0), ax_top.c2p(x1, y1), color=RED_B, stroke_width=3.2)

            # x_1 och x_2 mitt i Δx-sträckan på båda x-axlarna
            tick_x_mid_top = ax_top.x_axis.get_tick(x_center, size=0.07)
            tick_x_mid_bot = ax_bot.x_axis.get_tick(x_center, size=0.07)
            lbl_x_top = MathTex(rf"x_{i+1}", font_size=20).next_to(
                ax_top.c2p(x_center, 0), DOWN, buff=0.08
            )
            lbl_x_bot = MathTex(rf"x_{i+1}", font_size=20).next_to(
                ax_bot.c2p(x_center, 0), DOWN, buff=0.08
            )

            # Nätt måttklam och nedflyttad beteckning för Δx på båda axlarna
            brace_dx_top = slim_h_brace(
                ax_top.c2p(x0, 0), ax_top.c2p(x1, 0), y_offset=-0.34, tick_h=0.05
            )
            lbl_dx_top = MathTex(r"\Delta x", font_size=19).next_to(brace_dx_top, DOWN, buff=0.04)

            brace_dx_bot = slim_h_brace(
                ax_bot.c2p(x0, 0), ax_bot.c2p(x1, 0), y_offset=-0.32, tick_h=0.05
            )
            lbl_dx_bot = MathTex(r"\Delta x", font_size=19).next_to(brace_dx_bot, DOWN, buff=0.04)

            lbl_dy = MathTex(rf"\Delta y_{i+1}", font_size=22, color=GREEN).next_to(step_v, RIGHT, buff=0.08)
            lbl_sec = MathTex(rf"k_{i+1} = \frac{{\Delta y_{i+1}}}{{\Delta x}}", font_size=22, color=RED_B).next_to(
                secant.get_center(), UL, buff=0.06
            )

            rect = Rectangle(
                width=ax_bot.x_axis.unit_size * dx,
                height=ax_bot.y_axis.unit_size * slope,
                fill_color=GREEN, fill_opacity=0.45,
                stroke_color=GREEN_B, stroke_width=2
            ).move_to(ax_bot.c2p(x_center, slope / 2))

            dash_slope = DashedLine(
                ax_bot.c2p(0, slope), ax_bot.c2p(x0, slope), color=RED_B, stroke_width=1.5
            )
            tick_slope = ax_bot.y_axis.get_tick(slope, size=0.08).set_color(RED_B)
            lbl_slope_y = MathTex(
                rf"f(x_{i+1}) \approx \frac{{\Delta y_{i+1}}}{{\Delta x}}",
                font_size=20, color=RED_B
            ).next_to(ax_bot.c2p(0, slope), LEFT, buff=0.1)
            lbl_area = MathTex(rf"\text{{Area}} = \Delta y_{i+1}", font_size=20, color=GREEN_A).move_to(rect.get_center())

            self.play(
                Create(step_h),
                FadeIn(tick_x_mid_top, lbl_x_top),
                Create(brace_dx_top),
                Write(lbl_dx_top)
            )
            self.play(Create(step_v), Write(lbl_dy))
            self.play(Create(secant), Write(lbl_sec))
            self.wait(0.6)

            self.play(
                ReplacementTransform(secant.copy(), rect),
                Create(dash_slope),
                FadeIn(tick_slope, lbl_slope_y, tick_x_mid_bot, lbl_x_bot),
                Create(brace_dx_bot),
                Write(lbl_dx_bot)
            )
            self.play(Write(lbl_area))
            self.wait(0.8)

            self.play(FadeOut(lbl_sec), run_time=0.3)
            dy_lines.add(step_v)

        # 5. Stapla de gröna dy-linjerna direkt på övre y-axeln mellan F(a) och F(b)
        stacked_dys = VGroup()
        current_y = F(a)
        for i in range(n_steps):
            x0 = a + i * dx
            dy_val = F(x0 + dx) - F(x0)
            seg = Line(
                ax_top.c2p(0, current_y),
                ax_top.c2p(0, current_y + dy_val),
                color=GREEN, stroke_width=5
            )
            stacked_dys.add(seg)
            current_y += dy_val

        self.play(TransformFromCopy(dy_lines, stacked_dys), run_time=1.3)
        self.wait(0.6)

        # 6. Matematisk sammanfattning inkl. Riemannsumman till höger (justerad något nedåt under rubriken)
        summary = VGroup(
            MathTex(r"\text{1. Sekantens lutning: } k_i = \frac{\Delta y_i}{\Delta x}", font_size=24, color=RED_B),
            MathTex(r"\text{2. Rektangelarea: } \frac{\Delta y_i}{\Delta x} \cdot \Delta x = \Delta y_i", font_size=24, color=GREEN_A),
            MathTex(r"\text{3. Total area: } \Delta y_1 + \Delta y_2 = F(b) - F(a)", font_size=24, color=WHITE),
            Line(LEFT * 2.1, RIGHT * 2.1, color=GRAY),
            MathTex(r"\text{Då } \Delta x \to 0 \text{ och } n \to \infty \text{:}", font_size=23, color=YELLOW),
            MathTex(r"\lim_{n \to \infty} \sum_{i=1}^{n} \,\Delta y_i = \lim_{n \to \infty} \sum_{i=1}^{n} f(x_i)\,\Delta x = F(b) - F(a)", font_size=24, color=YELLOW_A),
            MathTex(r"\int_a^b f(x)\,dx = F(b) - F(a)", font_size=30, color=BLUE_B)
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.22).move_to(RIGHT * 3.6 + DOWN * 0.25)

        for line in summary:
            self.play(Write(line), run_time=0.6)
            self.wait(0.25)

        self.wait(2.5)

Manim Community v0.22.1